# Пример - Граничные данные и цепочки ядер

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **a7fa4b5**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/a7fa4b54aaa29f5c42213bf2e97f74f922da3ead.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


В 13 расчётах проверим, как граничные данные позволяют вычислить свойства сочленённого графа. Для четырёхточечной границы получим ранг 14, для двух цепочек из трёх шестиугольников — по 204 остовных дерева, а для двух способов замыкания квадратной полосы — 75 и 81.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra a7fa4b5](https://github.com/magin67/polyform_algebra/commit/a7fa4b54aaa29f5c42213bf2e97f74f922da3ead).


## Лесной паспорт пути

На пути $a-b-c-d-e$ с границей $a,c,e$ вычисляем полиномы по состояниям. Состояние $ae|c$ отсутствует, а сумма остальных полиномов даёт обычный лесной полином.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%81%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D0%BD%D0%B8%D1%8F---%D1%87%D1%82%D0%BE-%D0%BD%D1%83%D0%B6%D0%BD%D0%BE-%D1%81%D0%BE%D1%85%D1%80%D0%B0%D0%BD%D0%B8%D1%82%D1%8C-%D0%BE%D1%82-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B#%D0%BF%D1%8F%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D0%BE%D0%B2-%D0%BF%D1%83%D1%82%D0%B8-p5).

In [3]:
from collections import Counter
from fractions import Fraction as Q
from itertools import product

from sympy import Matrix, Symbol, expand

from polyform_algebra.boundary_data import (
    forest_passport, join_forest_states, pair_forest_passports,
    pair_spectral_passports, spectral_pair_passport,
)
from polyform_algebra.core import circumsphere, contract, kron_reduce, laplacian, network, spanning_tree_weight
from polyform_algebra.core_chains import cycle_chain_tree_count
from polyform_algebra.matroid import forest_coefficients
from polyform_algebra.toponomes import forest_toponome, laplacian_polynomial, rooted_forest_weight


def unit_graph(labels, edges):
    return network(tuple(labels), [(a, b, 1) for a, b in edges])


def path_edges(labels):
    return tuple(zip(labels, labels[1:]))


def cycle_edges(labels):
    return (*path_edges(labels), (labels[-1], labels[0]))


def polynomial(coefficients, variable):
    return sum(value * variable**grade for grade, value in enumerate(coefficients))


def state(name):
    return tuple(tuple(block) for block in name.split("|"))


path = unit_graph("abcde", path_edges("abcde"))
boundary = tuple("ace")
passport = forest_passport(path, boundary)
assert passport == {
    state("a|c|e"): {0: 1, 1: 4, 2: 4},
    state("ac|e"): {2: 1, 3: 2},
    state("a|ce"): {2: 1, 3: 2},
    state("ace"): {4: 1},
}
assert tuple(sum(poly.get(k, 0) for poly in passport.values()) for k in range(5)) == forest_coefficients(path)
for blocks, coefficients in passport.items():
    print("P5 state", "|".join(map("".join, blocks)),
          polynomial(tuple(coefficients.get(k, 0) for k in range(5)), Symbol("t")))

P5 state a|c|e 4*t**2 + 4*t + 1
P5 state ac|e 2*t**3 + t**2
P5 state ace t**4
P5 state a|ce 2*t**3 + t**2


## Три стягивания

Для каждого стягивания оставляем совместимые лесные состояния и сравниваем коэффициенты с прямым расчётом на фактор-графе.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D1%82%D1%8F%D0%B3%D0%B8%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD-%D0%BA%D0%B0%D0%BA-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B9-%D1%82%D0%B5%D1%81%D1%82#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D1%82%D1%80%D0%B8-%D1%81%D1%82%D1%8F%D0%B3%D0%B8%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%BF%D1%83%D1%82%D0%B8-p5).

In [4]:
for partition, expected in (
    ("ac|e", (1, 4, 5, 2)),
    ("ae|c", (1, 4, 6, 4)),
    ("ace", (1, 4, 4)),
):
    boundary_state = state(partition)
    quotient = contract(path, {"".join(block): block for block in (*boundary_state, ("b",), ("d",))}).value.network
    predicted = tuple(sum(poly.get(k, 0) for blocks, poly in passport.items()
                          if join_forest_states(boundary, blocks, boundary_state) is not None)
                      for k in range(len(quotient.context.labels)))
    assert predicted == forest_coefficients(quotient) == expected
    print("Contraction", partition, tuple(map(str, predicted)))

Contraction ac|e ('1', '4', '5', '2')
Contraction ae|c ('1', '4', '6', '4')
Contraction ace ('1', '4', '4')


## Спаривание лесных паспортов

Склеиваем два пути по общей границе $a,c,e$. Сравниваем все полиномы состояний, полученные свёрткой совместимых пар, с прямым перебором лесов.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82-%D0%BA%D0%B0%D0%BA-%D1%81%D0%BF%D0%B0%D1%80%D0%B8%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D1%85-%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85#%D0%B1%D0%B8%D0%BB%D0%B8%D0%BD%D0%B5%D0%B9%D0%BD%D1%8B%D0%B9-%D0%B7%D0%B0%D0%BA%D0%BE%D0%BD-%D0%BF%D0%BE%D0%BB%D0%BD%D0%BE%D0%B3%D0%BE-%D0%BB%D0%B5%D1%81%D0%BD%D0%BE%D0%B3%D0%BE-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D0%B0).

In [5]:
other_path = unit_graph("afcge", path_edges("afcge"))
joined = network(tuple("abcdefg"), path.edges + other_path.edges)
combined = pair_forest_passports(boundary, passport, forest_passport(other_path, boundary))
assert combined == forest_passport(joined, boundary)
assert combined[state("ace")][6] == 16
print("Paired passport:", len(combined), "states; spanning trees:", combined[state("ace")][6])

Paired passport: 4 states; spanning trees: 16


## Остовной слой и отклик

Для одного и двух путей сравниваем остовной кофактор с произведением внутреннего минора и кофактора граничного отклика.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%BE%D0%B9-%D1%81%D0%BB%D0%BE%D0%B9-%D0%B8-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D0%B0%D1%8F-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F#%D0%B4%D0%B2%D0%B0-%D0%BF%D1%83%D1%82%D0%B8-%D1%81-%D0%BE%D0%B1%D1%89%D0%B5%D0%B9-%D1%82%D1%80%D1%91%D1%85%D1%82%D0%BE%D1%87%D0%B5%D1%87%D0%BD%D0%BE%D0%B9-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D0%B5%D0%B9).

In [6]:
for graph, expected in ((path, 1), (joined, 16)):
    response = Matrix(laplacian(kron_reduce(graph, boundary).value.network).value.values)
    interior_minor = rooted_forest_weight(graph, boundary)
    assert interior_minor == (4 if graph == path else 16)
    assert interior_minor * response[1:, 1:].det() == spanning_tree_weight(graph).value == expected
assert Matrix(laplacian(kron_reduce(path, boundary).value.network).value.values) == Matrix(
    [[Q(1, 2), Q(-1, 2), 0], [Q(-1, 2), 1, Q(-1, 2)], [0, Q(-1, 2), Q(1, 2)]]
)
print("Spanning-tree counts via boundary response: 1, 16")

Spanning-tree counts via boundary response: 1, 16


## Ранг четырёхточечной матрицы

Строим матрицу совместимости для всех 15 разбиений. Проверяем вектор соотношения и определитель минора по первым 14 состояниям.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85%D1%82%D0%BE%D1%87%D0%B5%D1%87%D0%BD%D1%8B%D0%B9-%D0%BF%D0%B0%D1%81%D0%BF%D0%BE%D1%80%D1%82-%D0%B8-%D1%81%D0%BE%D0%BA%D1%80%D0%B0%D1%89%D0%B5%D0%BD%D0%B8%D0%B5-%D1%87%D0%B8%D1%81%D0%BB%D0%B0-%D1%81%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D0%BD%D0%B8%D0%B9#%D1%81%D0%BE%D0%BA%D1%80%D0%B0%D1%89%D1%91%D0%BD%D0%BD%D0%B0%D1%8F-%D0%BC%D0%B0%D1%82%D1%80%D0%B8%D1%86%D0%B0-%D0%B8-%D1%80%D0%B0%D0%BD%D0%B3-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D0%BD%D0%B0%D0%B4%D1%86%D0%B0%D1%82%D1%8C).

In [7]:
four_boundary = tuple("abcd")
names = ("a|b|c|d", "ab|c|d", "ac|b|d", "ad|b|c", "a|bc|d", "a|bd|c", "a|b|cd",
         "abc|d", "abd|c", "acd|b", "a|bcd", "ab|cd", "ad|bc", "abcd", "ac|bd")
states = tuple(map(state, names))
pairing = Matrix([[int(join_forest_states(four_boundary, a, b) is not None)
                   for b in states] for a in states])
relation = Matrix([0] * 7 + [-1] * 4 + [1] * 4)
assert pairing * relation == Matrix.zeros(15, 1)
assert pairing.rank() == 14 and pairing[:14, :14].det() == -64
print("Four-point matrix: rank", pairing.rank(), "out of", pairing.rows)

Four-point matrix: rank 14 out of 15


## Спектральное спаривание

Соединяем два пути длины два по концам. Сравниваем все пять полиномов с прямыми минорами и проверяем тождество Якоби. Равенство сохраняется и при $x=2$, где внутренний минор обращается в ноль.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BE%D1%82-%D0%BB%D0%B5%D1%81%D0%BD%D0%BE%D0%B3%D0%BE-%D0%BF%D0%B0%D1%81%D0%BF%D0%BE%D1%80%D1%82%D0%B0-%D0%BA-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%BC-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%BC-%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%BC#%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BF%D0%BE-%D0%B4%D0%B2%D1%83%D0%BC-%D0%BE%D0%B1%D1%89%D0%B8%D0%BC-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D0%BC).

In [8]:
x = Symbol("x")
terminals = tuple("ab")
first = unit_graph("acb", path_edges("acb"))
second = unit_graph("adb", path_edges("adb"))
cycle = network(tuple("acbd"), first.edges + second.edges)
left, right = (spectral_pair_passport(graph, terminals, x) for graph in (first, second))
full, minor_a, minor_b, cross, interior = left
assert expand(full * interior - minor_a * minor_b + cross**2) == 0
paired = pair_spectral_passports(left, right, x)
assert paired == spectral_pair_passport(cycle, terminals, x)
assert paired[0] == x**4 - 8*x**3 + 20*x**2 - 16*x
assert paired[0].subs(x, 2) == 0 and interior.subs(x, 2) == 0
print("Spectral gluing:", paired[0])

Spectral gluing:

 x**4 - 8*x**3 + 20*x**2 - 16*x


## Остовы квадратной лестницы

Сравниваем рекурсию $\tau_{k+2}=4\tau_{k+1}-\tau_k$ с прямыми остовными кофакторами лестниц.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D0%BF%D0%BB%D0%B5%D1%82%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2---%D0%B2%D0%B8%D0%BB%D0%BA%D0%B0,-%D1%80%D0%BE%D0%B7%D0%B5%D1%82%D0%BA%D0%B0-%D0%B8-%D1%80%D0%B5%D0%BA%D1%83%D1%80%D1%81%D0%B8%D1%8F#%D1%81%D0%BD%D0%B0%D1%87%D0%B0%D0%BB%D0%B0-%D0%BA%D0%B2%D0%B0%D0%B4%D1%80%D0%B0%D1%82%D0%BD%D0%B0%D1%8F-%D0%BB%D0%B5%D1%81%D1%82%D0%BD%D0%B8%D1%86%D0%B0).

In [9]:
def ladder(top, bottom):
    return unit_graph(top + bottom, (*path_edges(top), *path_edges(bottom), *zip(top, bottom)))


top, bottom = "abcdef", "uvwxyz"
ladders = [ladder(top[:n], bottom[:n]) for n in range(1, 7)]
direct = [spanning_tree_weight(graph).value for graph in ladders]
recurrence = [1, 4]
for _ in range(4):
    recurrence.append(4 * recurrence[-1] - recurrence[-2])
assert direct == recurrence == [1, 4, 15, 56, 209, 780]
print("Square ladder tree counts:", tuple(map(str, direct)))

Square ladder tree counts: ('1', '4', '15', '56', '209', '780')


## Перенос лесных состояний лестницы

Сравниваем перенос двух полиномов для соединённых и разъединённых контактов с паспортами лестниц. На одном квадрате также проверяем топоном по размерам компонент и корневой полином.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D0%B0-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0-%D0%BF%D1%80%D0%B8-%D0%BF%D0%BB%D0%B5%D1%82%D0%B5%D0%BD%D0%B8%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2#%D1%82%D0%BE%D1%87%D0%BD%D1%8B%D0%B5-%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B8-%D0%B8-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D1%8B-%D1%80%D0%B5%D0%B7%D1%83%D0%BB%D1%8C%D1%82%D0%B0%D1%82%D0%B0).

In [10]:
t = Symbol("t")
forest_polynomials = [polynomial(forest_coefficients(graph), t) for graph in ladders[:4]]
assert [f.subs(t, 1) for f in forest_polynomials] == [2, 15, 112, 836]
h4, b4 = (1 + t)**2, t**2
transfer = Matrix([[t*h4 + (1-t)*b4, t*h4], [h4-b4, h4]])
predicted = Matrix([t, 1])
for index, graph in enumerate(ladders[:4]):
    active = (top[index], bottom[index])
    data = forest_passport(graph, active)
    observed = Matrix([sum(value*t**k for k, value in data[blocks].items())
                       for blocks in ((active,), tuple((v,) for v in active))])
    assert (observed - predicted).applyfunc(expand) == Matrix.zeros(2, 1)
    predicted = transfer * predicted
for f, g, h in zip(forest_polynomials, forest_polynomials[1:], forest_polynomials[2:]):
    assert expand(h - (1 + 3*t + 4*t**2)*g + t**2*(1+t)**2*f) == 0
assert forest_toponome(ladders[1]) == {(): 1, (1,): 4, (2,): 4, (1, 1): 2, (3,): 4}
assert laplacian_polynomial(ladders[1]) == (1, 8, 20, 16)
print("Square ladder forest counts:", tuple(f.subs(t, 1) for f in forest_polynomials))

Square ladder forest counts: (2, 15, 112, 836)


## Две допустимые цепочки

Собираем две цепочки с разным расположением шва. Общие рёбра учитываем один раз. Сверяем число вершин, рёбер, остовных деревьев и резистивный радиус.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D1%8F%D0%B4%D1%80%D0%B0,-%D1%88%D0%B2%D1%8B-%D0%B8-%D0%B4%D0%BE%D0%BF%D1%83%D1%81%D1%82%D0%B8%D0%BC%D1%8B%D0%B5-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%BA%D0%B8#%D0%B4%D0%B2%D0%B5-%D1%81%D0%B1%D0%BE%D1%80%D0%BA%D0%B8-%D1%82%D1%80%D1%91%D1%85-%D1%88%D0%B5%D1%81%D1%82%D0%B8%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2).

In [11]:
def chain_of_cycles(cores):
    labels = tuple(dict.fromkeys("".join(cores)))
    edges = sorted({tuple(sorted(edge)) for core in cores for edge in cycle_edges(core)})
    return unit_graph(labels, edges)


def hexagon_chain(second_seam):
    cores = ("abjihg", "abcdef", second_seam + "nmlk")
    first_seam = set("ab")
    assert set(cores[0]) & set(cores[1]) == first_seam
    assert set(cores[1]) & set(cores[2]) == set(second_seam)
    assert not set(cores[0]) & set(cores[2])
    graph = chain_of_cycles(cores)
    assert len(graph.context.labels) == 14 and len(graph.edges) == 16
    return graph


hexagons = (hexagon_chain("de"), hexagon_chain("cd"))
assert [spanning_tree_weight(graph).value for graph in hexagons] == [204, 204]
assert [circumsphere(graph).value.resistance_radius for graph in hexagons] == [Q(43595, 41616)] * 2
print("Three hexagons: 14 vertices, 16 edges, 204 trees in each assembly")

Three hexagons: 14 vertices, 16 edges, 204 trees in each assembly


## Матроид и конфигурация

Сравниваем кратности рёбер двойственных графов по парам граней. Обычные лесные полиномы совпадают, а корневые различаются.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D0%BF%D0%BE%D1%87%D0%B5%D0%BC%D1%83-%D0%BA%D0%BE%D0%BD%D1%84%D0%B8%D0%B3%D1%83%D1%80%D0%B0%D1%86%D0%B8%D1%8F-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%BA%D0%B8-%D1%81%D0%BE%D1%85%D1%80%D0%B0%D0%BD%D1%8F%D0%B5%D1%82-%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B#%D0%B4%D0%B2%D0%B5-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%BA%D0%B8-%D0%B8%D0%B7-%D1%82%D1%80%D1%91%D1%85-%D1%88%D0%B5%D1%81%D1%82%D0%B8%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2).

In [12]:
def dual_edge_counts(cores):
    # Face A denotes the outer face; B, C, D are the three cycle cores.
    faces = {}
    for face, core in zip("BCD", cores, strict=True):
        for edge in cycle_edges(core):
            faces.setdefault(tuple(sorted(edge)), []).append(face)
    return Counter(tuple(sorted(pair if len(pair) == 2 else ["A", *pair])) for pair in faces.values())


dual_counts = [dual_edge_counts(("abjihg", "abcdef", seam + "nmlk")) for seam in ("de", "cd")]
assert dual_counts[0] == dual_counts[1] == {("A", "B"): 5, ("A", "C"): 4, ("A", "D"): 5,
                                           ("B", "C"): 1, ("C", "D"): 1}
ordinary = [forest_coefficients(graph) for graph in hexagons]
rooted = [laplacian_polynomial(graph) for graph in hexagons]
assert ordinary[0] == ordinary[1]
assert rooted[0] != rooted[1]
degree_three_edges = []
for graph in hexagons:
    degree_three = {v for v in graph.context.labels if sum(v in edge[:2] for edge in graph.edges) == 3}
    degree_three_edges.append(sum(a in degree_three and b in degree_three for a, b, _ in graph.edges))
assert degree_three_edges == [2, 3]
print("Hexagon configurations: same ordinary polynomial; degree-three edges:", *degree_three_edges)

Hexagon configurations: same ordinary polynomial; degree-three edges: 2 3


## Определитель и континуант

Сравниваем континуант с определителем двойственного минора и с остовными кофакторами явных цепочек.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%B5%D0%BA---%D0%BE%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D0%B8%D1%82%D0%B5%D0%BB%D1%8C-%D0%B8-%D1%80%D0%B5%D0%BA%D1%83%D1%80%D1%81%D0%B8%D1%8F#%D1%8F%D0%B2%D0%BD%D1%8B%D0%B5-%D1%81%D0%B1%D0%BE%D1%80%D0%BA%D0%B8-%D0%B8-%D0%BD%D0%B5%D0%B7%D0%B0%D0%B2%D0%B8%D1%81%D0%B8%D0%BC%D1%8B%D0%B5-%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B8).

In [13]:
for size in (4, 6):
    for count in range(1, 6):
        dual_minor = Matrix(count, count, lambda i, j: size if i == j else -int(abs(i-j) == 1))
        assert cycle_chain_tree_count([size] * count, [1] * (count-1)) == dual_minor.det()
assert cycle_chain_tree_count([4, 5, 6], [1, 2]) == cycle_chain_tree_count([6, 5, 4], [2, 1])
assert cycle_chain_tree_count([6, 6, 6], [1, 1]) == spanning_tree_weight(hexagons[0]).value
assert cycle_chain_tree_count([4] * 5, [1] * 4) == direct[-1]
print("Continuants agree with dual determinants and direct cofactors")

Continuants agree with dual determinants and direct cofactors


## Границы остовного счёта

Проверяем два графа с 12 остовными деревьями. Точные резистивные радиусы различаются.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D1%8B-%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D0%BD%D0%BE%D1%81%D1%82%D0%B8---%D0%B2%D0%B5%D1%82%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5,-%D0%BD%D0%B5%D1%81%D0%B2%D1%8F%D0%B7%D0%BD%D1%8B%D0%B5-%D1%88%D0%B2%D1%8B-%D0%B8-%D0%B4%D1%80%D1%83%D0%B3%D0%B8%D0%B5-%D1%8F%D0%B4%D1%80%D0%B0#%D0%BE%D0%B4%D0%B8%D0%BD%D0%B0%D0%BA%D0%BE%D0%B2%D0%BE%D0%B5-%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%BE%D0%B5-%D1%87%D0%B8%D1%81%D0%BB%D0%BE-%D0%BD%D0%B5-%D0%BE%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D1%8F%D0%B5%D1%82-%D1%80%D0%B0%D0%B4%D0%B8%D1%83%D1%81).

In [14]:
square_triangle = chain_of_cycles(("abcd", "aef"))
bipartite_with_leaf = unit_graph("abcdef", (*product("ab", "cde"), ("a", "f")))
assert [spanning_tree_weight(graph).value for graph in (square_triangle, bipartite_with_leaf)] == [12, 12]
radii = [circumsphere(graph).value.resistance_radius for graph in (square_triangle, bipartite_with_leaf)]
assert radii == [Q(77, 144), Q(7, 12)]
print("Equal tree counts, different radii:", *radii)

Equal tree counts, different radii: 77/144 7/12


## Два замыкания полосы

Добавляем прямой или перевёрнутый шов. Сравниваем остовные числа и радиусы цилиндрической полосы и полосы Мёбиуса.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B5%D0%BF%D0%B8-%D0%B8%D0%B7-%D1%8F%D0%B4%D0%B5%D1%80/%D0%B7%D0%B0%D0%BC%D1%8B%D0%BA%D0%B0%D0%BD%D0%B8%D0%B5-%D1%86%D0%B5%D0%BF%D0%B8---%D0%BA%D0%BE%D0%BB%D1%8C%D1%86%D0%BE-%D0%B8-%D1%8D%D1%84%D1%84%D0%B5%D0%BA%D1%82-%D0%BC%D1%91%D0%B1%D0%B8%D1%83%D1%81%D0%B0#%D0%BF%D0%BE%D1%81%D0%BB%D0%B5%D0%B4%D0%BD%D0%B8%D0%B9-%D1%88%D0%BE%D0%B2-%D0%BA%D0%B2%D0%B0%D0%B4%D1%80%D0%B0%D1%82%D0%BD%D0%BE%D0%B9-%D0%BF%D0%BE%D0%BB%D0%BE%D1%81%D1%8B).

In [15]:
strip_top, strip_bottom = "abc", "uvw"
rungs_and_rails = (*path_edges(strip_top), *path_edges(strip_bottom), *zip(strip_top, strip_bottom))
cylinder = unit_graph(strip_top + strip_bottom, (*rungs_and_rails, ("c", "a"), ("w", "u")))
mobius = unit_graph(strip_top + strip_bottom, (*rungs_and_rails, ("c", "u"), ("w", "a")))
counts = [spanning_tree_weight(graph).value for graph in (cylinder, mobius)]
radii = [circumsphere(graph).value.resistance_radius for graph in (cylinder, mobius)]
assert counts == [75, 81] and counts[1] - counts[0] == 2 * 3
assert radii == [Q(47, 180), Q(1, 4)]
print("Cylindrical and Möbius closures:", *counts)

Cylindrical and Möbius closures: 75 81
